# Lab 1: Problem-solving agents and search problem formulation

## Learning goals
By the end of this lab, students can:
- explain the four phases of a problem-solving agent: goal formulation, problem formulation, search, and execution;
- define the five parts of a search problem: initial state, actions, transition model, goal test, and action cost;
- run a route-finding agent on the Romania map;
- step through a GUI map showing the agent's planned route and execution sequence.

Source note: The notes in this notebook are paraphrased from Chapter 3, Solving Problems by Searching, in the uploaded course document. No outside notes are used.


## Chapter 3 notes for this lab

A problem-solving agent plans ahead when the best immediate action is not obvious. It uses an abstract, atomic representation of states. For the Romania route problem, a state can simply be the current city. The agent uses a known map to search for a sequence of actions that reaches a goal city.

A well-defined search problem contains:
1. an initial state;
2. available actions for each state;
3. a transition model showing the result of each action;
4. a goal test;
5. an action cost function.

A solution is a path from the initial state to a goal state. An optimal solution is the lowest-cost solution path.


In [1]:
# Run this cell first.
# These notebooks use only local Python code plus matplotlib and ipywidgets.
# If a student machine is missing a package, uncomment and run the next line once:
# %pip install matplotlib ipywidgets

import math
import heapq
import itertools
from collections import deque
from html import escape

import matplotlib.pyplot as plt

# HTML display works even if ipywidgets is not installed.
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception as exc:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Install with: pip install ipywidgets")


In [2]:
ROMANIA_GRAPH = {
    "Arad": {"Zerind": 75, "Sibiu": 140, "Timisoara": 118},
    "Zerind": {"Arad": 75, "Oradea": 71},
    "Oradea": {"Zerind": 71, "Sibiu": 151},
    "Sibiu": {"Arad": 140, "Oradea": 151, "Fagaras": 99, "Rimnicu Vilcea": 80},
    "Timisoara": {"Arad": 118, "Lugoj": 111},
    "Lugoj": {"Timisoara": 111, "Mehadia": 70},
    "Mehadia": {"Lugoj": 70, "Drobeta": 75},
    "Drobeta": {"Mehadia": 75, "Craiova": 120},
    "Craiova": {"Drobeta": 120, "Rimnicu Vilcea": 146, "Pitesti": 138},
    "Rimnicu Vilcea": {"Sibiu": 80, "Craiova": 146, "Pitesti": 97},
    "Fagaras": {"Sibiu": 99, "Bucharest": 211},
    "Pitesti": {"Rimnicu Vilcea": 97, "Craiova": 138, "Bucharest": 101},
    "Bucharest": {"Fagaras": 211, "Pitesti": 101, "Giurgiu": 90, "Urziceni": 85},
    "Giurgiu": {"Bucharest": 90},
    "Urziceni": {"Bucharest": 85, "Hirsova": 98, "Vaslui": 142},
    "Hirsova": {"Urziceni": 98, "Eforie": 86},
    "Eforie": {"Hirsova": 86},
    "Vaslui": {"Urziceni": 142, "Iasi": 92},
    "Iasi": {"Vaslui": 92, "Neamt": 87},
    "Neamt": {"Iasi": 87},
}

# Coordinates are fixed local drawing coordinates, not GPS coordinates.
ROMANIA_POS = {
    "Arad": (91, 492), "Zerind": (108, 531), "Oradea": (123, 567),
    "Sibiu": (207, 457), "Timisoara": (94, 410), "Lugoj": (165, 379),
    "Mehadia": (168, 339), "Drobeta": (165, 299), "Craiova": (253, 288),
    "Rimnicu Vilcea": (233, 410), "Fagaras": (305, 449), "Pitesti": (320, 368),
    "Bucharest": (400, 327), "Giurgiu": (375, 270), "Urziceni": (456, 350),
    "Hirsova": (534, 350), "Eforie": (562, 293), "Vaslui": (509, 444),
    "Iasi": (473, 506), "Neamt": (406, 537),
}

# Straight-line distance estimates to Bucharest, used only when Bucharest is the goal.
H_SLD_BUCHAREST = {
    "Arad": 366, "Bucharest": 0, "Craiova": 160, "Drobeta": 242, "Eforie": 161,
    "Fagaras": 176, "Giurgiu": 77, "Hirsova": 151, "Iasi": 226, "Lugoj": 244,
    "Mehadia": 241, "Neamt": 234, "Oradea": 380, "Pitesti": 100,
    "Rimnicu Vilcea": 193, "Sibiu": 253, "Timisoara": 329,
    "Urziceni": 80, "Vaslui": 199, "Zerind": 374,
}

def sld_to_bucharest(city):
    return H_SLD_BUCHAREST.get(city, 0)

def edge_list(graph):
    seen = set()
    edges = []
    for a, nbrs in graph.items():
        for b, cost in nbrs.items():
            key = tuple(sorted((a, b)))
            if key not in seen:
                seen.add(key)
                edges.append((a, b, cost))
    return edges


In [3]:
_NODE_COUNTER = itertools.count()

class Node:
    """Search-tree node: state, parent, action, path cost g(n), depth, and order for tie breaking."""
    def __init__(self, state, parent=None, action=None, path_cost=0, depth=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = depth
        self.order = next(_NODE_COUNTER)

    def __repr__(self):
        return f"Node(state={self.state!r}, g={self.path_cost}, depth={self.depth})"

def reset_node_counter():
    global _NODE_COUNTER
    _NODE_COUNTER = itertools.count()

def path_nodes(node):
    nodes = []
    while node is not None:
        nodes.append(node)
        node = node.parent
    return list(reversed(nodes))

def path_states(node):
    return [n.state for n in path_nodes(node)]

def path_actions(node):
    return [n.action for n in path_nodes(node)[1:]]

def path_cost_of_states(graph, states):
    total = 0
    for a, b in zip(states, states[1:]):
        total += graph[a][b]
    return total

def states_to_edges(states):
    return {tuple(sorted((a, b))) for a, b in zip(states, states[1:])}

def make_html_table(headers, rows, max_rows=15):
    rows = rows[:max_rows]
    html = "<table style='border-collapse:collapse; font-family:monospace;'>"
    html += "<tr>" + "".join(f"<th style='border:1px solid #999; padding:4px 8px;'>{escape(str(h))}</th>" for h in headers) + "</tr>"
    for row in rows:
        html += "<tr>" + "".join(f"<td style='border:1px solid #999; padding:4px 8px;'>{escape(str(v))}</td>" for v in row) + "</tr>"
    html += "</table>"
    return HTML(html)

def draw_romania_map(step=None, title="Romania search map", heuristic=None, f_func=None):
    """Draw a step of search on the Romania state-space graph."""
    step = step or {}
    expanded = set(step.get("expanded", []))
    expanded_b = set(step.get("expanded_b", []))
    generated = set(step.get("generated", []))
    frontier_nodes = step.get("frontier_nodes", [])
    frontier = {n.state if hasattr(n, "state") else n for n in frontier_nodes}
    frontier_b_nodes = step.get("frontier_b_nodes", [])
    frontier_b = {n.state if hasattr(n, "state") else n for n in frontier_b_nodes}
    current = step.get("current", None)
    current_state = current.state if hasattr(current, "state") else current
    current_b = step.get("current_b", None)
    current_b_state = current_b.state if hasattr(current_b, "state") else current_b

    solution_path = step.get("solution_path", []) or []
    current_path = path_states(current) if hasattr(current, "parent") else []
    solution_edges = states_to_edges(solution_path)
    current_edges = states_to_edges(current_path)

    fig, ax = plt.subplots(figsize=(12, 7))

    # Draw edges first.
    for a, b, cost in edge_list(ROMANIA_GRAPH):
        x1, y1 = ROMANIA_POS[a]
        x2, y2 = ROMANIA_POS[b]
        edge_key = tuple(sorted((a, b)))
        if edge_key in solution_edges:
            ax.plot([x1, x2], [y1, y2], linewidth=5, color="#2ca02c", zorder=1)
        elif edge_key in current_edges:
            ax.plot([x1, x2], [y1, y2], linewidth=4, color="#ff7f0e", zorder=1)
        else:
            ax.plot([x1, x2], [y1, y2], linewidth=1, color="#999999", zorder=0)
        ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(cost), fontsize=8, color="#444444")

    # Draw nodes.
    for city, (x, y) in ROMANIA_POS.items():
        color = "#f7f7f7"
        size = 320
        edgecolor = "#333333"
        linewidth = 1.0
        if city in expanded:
            color = "#d62728"
            size = 410
        if city in expanded_b:
            color = "#9467bd"
            size = 410
        if city in frontier:
            color = "#1f77b4"
            size = 450
        if city in frontier_b:
            color = "#17becf"
            size = 450
        if city in generated:
            color = "#ffbb78"
            size = 470
        if city == current_state or city == current_b_state:
            color = "#ff7f0e"
            size = 560
            linewidth = 2.5
        if city in solution_path:
            edgecolor = "#2ca02c"
            linewidth = 3.0
        ax.scatter([x], [y], s=size, c=color, edgecolors=edgecolor, linewidths=linewidth, zorder=3)
        label = city
        if heuristic is not None:
            label += f"\nh={heuristic(city)}"
        ax.text(x, y - 18, label, ha="center", va="top", fontsize=8)

    legend_lines = [
        "Gray edge: road with action cost",
        "Blue: frontier; red: expanded; orange: current/generated",
        "Green outline/path: solution path when found",
    ]
    if step.get("direction"):
        legend_lines.append(f"Bidirectional step direction: {step['direction']}")
    ax.text(0.01, 0.01, "\n".join(legend_lines), transform=ax.transAxes, fontsize=9,
            bbox=dict(facecolor="white", alpha=0.9, edgecolor="#cccccc"))
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.show()

def step_summary_html(step, heuristic=None, f_func=None, max_rows=12):
    current = step.get("current")
    current_state = current.state if hasattr(current, "state") else current
    msg = step.get("message", "")
    path = path_states(current) if hasattr(current, "parent") else []
    text = f"<b>Step {step.get('step', 0)}.</b> {escape(msg)}<br>"
    if current_state is not None:
        text += f"<b>Current:</b> {escape(str(current_state))}<br>"
    if path:
        text += f"<b>Current path:</b> {' -> '.join(escape(str(s)) for s in path)}"
        if hasattr(current, "path_cost"):
            text += f" | g(n)={current.path_cost}"
        text += "<br>"

    rows = []
    for n in step.get("frontier_nodes", [])[:max_rows]:
        h = heuristic(n.state) if heuristic else 0
        f = f_func(n) if f_func else ""
        rows.append([n.state, n.depth, n.path_cost, h, f])
    table = ""
    if rows:
        table = make_html_table(["state", "depth", "g(n)", "h(n)", "f(n)"], rows, max_rows=max_rows).data
    return HTML(text + table)

def make_stepper(steps, draw_func, table_func=None, title="Step-through viewer"):
    if not steps:
        print("No steps to show.")
        return

    if not WIDGETS_AVAILABLE:
        print("ipywidgets is not available, so only the final step is shown.")
        draw_func(steps[-1])
        if table_func:
            display(table_func(steps[-1]))
        return

    slider = widgets.IntSlider(value=0, min=0, max=len(steps) - 1, step=1, description="Step")
    previous_button = widgets.Button(description="Previous")
    next_button = widgets.Button(description="Next")
    play = widgets.Play(value=0, min=0, max=len(steps) - 1, step=1, interval=1200, description="Play")
    widgets.jslink((play, "value"), (slider, "value"))
    out = widgets.Output()

    def render(*args):
        with out:
            clear_output(wait=True)
            step = steps[slider.value]
            draw_func(step)
            if table_func:
                display(table_func(step))

    def go_previous(_):
        slider.value = max(slider.min, slider.value - 1)

    def go_next(_):
        slider.value = min(slider.max, slider.value + 1)

    previous_button.on_click(go_previous)
    next_button.on_click(go_next)
    slider.observe(render, names="value")
    display(HTML(f"<h3>{escape(title)}</h3>"))
    display(widgets.HBox([previous_button, next_button, play, slider]))
    display(out)
    render()


## Build a reusable search-problem class


In [4]:
class RouteFindingProblem:
    """A Chapter 3 style search problem for the Romania map."""
    def __init__(self, graph, initial, goal):
        self.graph = graph
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        return sorted(self.graph[state].keys())

    def result(self, state, action):
        # In this formulation, the action name is the destination city.
        if action not in self.graph[state]:
            raise ValueError(f"Cannot go from {state} to {action}.")
        return action

    def is_goal(self, state):
        return state == self.goal

    def action_cost(self, state, action, next_state):
        return self.graph[state][next_state]

    def describe(self):
        return {
            "initial_state": self.initial,
            "goal_state": self.goal,
            "example_actions_from_initial": self.actions(self.initial),
            "state_count": len(self.graph),
            "road_count": len(edge_list(self.graph)),
        }

problem = RouteFindingProblem(ROMANIA_GRAPH, initial="Arad", goal="Bucharest")
problem.describe()


{'initial_state': 'Arad',
 'goal_state': 'Bucharest',
 'example_actions_from_initial': ['Sibiu', 'Timisoara', 'Zerind'],
 'state_count': 20,
 'road_count': 23}

## Implement the problem-solving agent cycle


In [5]:
def uniform_cost_plan(problem):
    """Uniform-cost search for route planning. It returns a Node or None."""
    reset_node_counter()
    root = Node(problem.initial)
    frontier = [(root.path_cost, root.order, root)]
    reached = {problem.initial: root}

    while frontier:
        _, _, node = heapq.heappop(frontier)
        if reached.get(node.state) is not node:
            continue
        if problem.is_goal(node.state):
            return node

        for action in problem.actions(node.state):
            next_state = problem.result(node.state, action)
            cost = problem.action_cost(node.state, action, next_state)
            child = Node(
                next_state,
                parent=node,
                action=f"Drive from {node.state} to {next_state}",
                path_cost=node.path_cost + cost,
                depth=node.depth + 1,
            )
            if next_state not in reached or child.path_cost < reached[next_state].path_cost:
                reached[next_state] = child
                heapq.heappush(frontier, (child.path_cost, child.order, child))
    return None

class ProblemSolvingAgent:
    """Goal formulation -> problem formulation -> search -> execution."""
    def __init__(self, graph):
        self.graph = graph
        self.current_state = None
        self.goal = None
        self.solution_node = None
        self.execution_index = 0

    def goal_formulation(self, current_state, goal):
        self.current_state = current_state
        self.goal = goal
        return f"Goal formulated: get from {current_state} to {goal}."

    def problem_formulation(self):
        return RouteFindingProblem(self.graph, self.current_state, self.goal)

    def search(self, problem):
        self.solution_node = uniform_cost_plan(problem)
        self.execution_index = 0
        return self.solution_node

    def execution_steps(self):
        if self.solution_node is None:
            return []
        states = path_states(self.solution_node)
        actions = path_actions(self.solution_node)
        rows = []
        for i, state in enumerate(states):
            action = "Start" if i == 0 else actions[i - 1]
            rows.append({"step": i, "state": state, "action_taken": action})
        return rows

agent = ProblemSolvingAgent(ROMANIA_GRAPH)
print(agent.goal_formulation("Arad", "Bucharest"))
route_problem = agent.problem_formulation()
solution = agent.search(route_problem)

print("Solution states:", " -> ".join(path_states(solution)))
print("Solution actions:")
for action in path_actions(solution):
    print(" -", action)
print("Total path cost:", solution.path_cost)


Goal formulated: get from Arad to Bucharest.
Solution states: Arad -> Sibiu -> Rimnicu Vilcea -> Pitesti -> Bucharest
Solution actions:
 - Drive from Arad to Sibiu
 - Drive from Sibiu to Rimnicu Vilcea
 - Drive from Rimnicu Vilcea to Pitesti
 - Drive from Pitesti to Bucharest
Total path cost: 418


## GUI 1: step through the four problem-solving phases


In [6]:
def agent_cycle_steps(agent, problem, solution_node):
    states = path_states(solution_node)
    steps = []

    root = Node(problem.initial)
    steps.append({
        "step": 0,
        "message": f"Goal formulation: choose the goal state {problem.goal}.",
        "current": root,
        "expanded": [],
        "frontier_nodes": [root],
        "generated": [],
        "solution_path": None,
    })

    steps.append({
        "step": 1,
        "message": "Problem formulation: model states as cities, actions as roads, and action costs as road distances.",
        "current": root,
        "expanded": [],
        "frontier_nodes": [root],
        "generated": list(problem.actions(problem.initial)),
        "solution_path": None,
    })

    steps.append({
        "step": 2,
        "message": "Search: uniform-cost search finds a lowest-cost route before acting in the real world.",
        "current": solution_node,
        "expanded": states,
        "frontier_nodes": [],
        "generated": [],
        "solution_path": states,
    })

    for i, city in enumerate(states):
        fake_node = Node(city, path_cost=path_cost_of_states(ROMANIA_GRAPH, states[:i+1]), depth=i)
        steps.append({
            "step": len(steps),
            "message": f"Execution: perform action {i}. The agent is now at {city}.",
            "current": fake_node,
            "expanded": states[:i+1],
            "frontier_nodes": [],
            "generated": [],
            "solution_path": states[:i+1],
        })
    return steps

steps = agent_cycle_steps(agent, route_problem, solution)
make_stepper(
    steps,
    draw_func=lambda step: draw_romania_map(step, title="Lab 1: problem-solving agent cycle"),
    table_func=lambda step: step_summary_html(step),
    title="Problem-solving agent GUI map"
)


Output()

## Student practice


In [ ]:
# Change the start city and rerun this cell.
start_city = "Timisoara"
goal_city = "Bucharest"

student_agent = ProblemSolvingAgent(ROMANIA_GRAPH)
print(student_agent.goal_formulation(start_city, goal_city))
student_problem = student_agent.problem_formulation()
student_solution = student_agent.search(student_problem)

print("Route:", " -> ".join(path_states(student_solution)))
print("Total cost:", student_solution.path_cost)

student_steps = agent_cycle_steps(student_agent, student_problem, student_solution)
make_stepper(
    student_steps,
    draw_func=lambda step: draw_romania_map(step, title=f"Route from {start_city} to {goal_city}"),
    table_func=lambda step: step_summary_html(step),
    title="Practice GUI map"
)
